In [ ]:
import glob

# Find all JPG images in the test images folder
image_files = glob.glob("/content/clothing--1/test/images/*.jpg")

if image_files:
    # Select the fifth image from the list
    single_image = image_files[4]

    # Run YOLO object detection on the selected image
    results = model.predict(
        source=single_image,
        conf=0.35,
        iou=0.50,
        save=True
    )

    # Build the path to the saved prediction result
    output_path = os.path.join(
        results[0].save_dir,
        os.path.basename(results[0].path)
    )

    # Display the predicted image
    display(Image(filename=output_path))

In [ ]:
import os
import glob
import cv2

# Search for all image files inside the project folder
patterns = [
    r"E:\Esso\Project\**\*.jpg",
    r"E:\Esso\Project\**\*.jpeg",
    r"E:\Esso\Project\**\*.png",
    r"E:\Esso\Project\**\*.JPG",
    r"E:\Esso\Project\**\*.JPEG",
    r"E:\Esso\Project\**\*.PNG"
]

# Create an empty list to store all image paths
image_files = []

# Search for images using all specified patterns
for pattern in patterns:
    image_files.extend(glob.glob(pattern, recursive=True))

if image_files:
    # Select the first image
    single_image = image_files[0]

    # Read the selected image
    image = cv2.imread(single_image)

    # Convert the image from BGR to RGB
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    print("Found:", len(image_files), "images")
    print("Selected:", single_image)
    print("Shape:", image.shape)

else:
    print("No images found anywhere inside E:\\Esso\\Project")

Found: 5346 images
Selected: E:\Esso\Project\clothing--1\clothing.v1-cloth.yolo26\test\images\100932_jpg.rf.3c4126472ec60ad757af7e378e8c5f5c.jpg
Shape: (600, 400, 3)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# Import NumPy for numerical operations
# Import Pandas for data manipulation and analysis
# Import KMeans for clustering data into groups

In [ ]:
def extract_top_colors(image_path, k=3):

    # 1) Read the image
    image_bgr = cv2.imread(image_path)

    # 2) Convert the image from BGR to RGB
    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB
    )

    # 3) Reshape the image into a list of pixels
    # Each pixel is represented as [R, G, B]
    pixels = image_rgb.reshape(-1, 3)

    # 4) Use K-Means to divide the pixels into k dominant colors
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(pixels)

    # 5) Get the center of each cluster as the RGB color
    colors = kmeans.cluster_centers_.astype(int)

    # 6) Count the number of pixels assigned to each cluster
    counts = np.bincount(kmeans.labels_)

    # 7) Convert the pixel counts into percentages
    percentages = counts / counts.sum() * 100

    # 8) Sort the colors from the most dominant to the least dominant
    order = np.argsort(percentages)[::-1]

    colors = colors[order]
    percentages = percentages[order]

    return colors, percentages

In [ ]:
cropped_files = glob.glob(
    "/content/cropped_items/*.jpg"
)

outfit_primary_colors = []

for image_path in cropped_files:

    # Extract the top 3 dominant colors
    colors, percentages = extract_top_colors(
        image_path,
        k=3
    )

    # The first color is the largest cluster, so it is the primary color
    primary_rgb = colors[0]
    primary_percentage = percentages[0]

    # Get the file name
    file_name = os.path.basename(image_path)

    # Extract the item name from the file name
    item_name = file_name.split("_")[0]

    # Store the result
    outfit_primary_colors.append({
        "item": item_name,
        "rgb": primary_rgb,
        "percentage": primary_percentage
    })

    print(
        f"{item_name}: "
        f"RGB {primary_rgb} "
        f"({primary_percentage:.1f}%)"
    )

In [ ]:
# Define a basic fashion color palette using RGB values
fashion_colors = {
    "Black": (0, 0, 0),
    "White": (255, 255, 255),
    "Dark Gray": (64, 64, 64),
    "Gray": (128, 128, 128),
    "Light Gray": (211, 211, 211),

    "Beige": (245, 245, 220),
    "Cream": (255, 253, 208),
    "Tan": (210, 180, 140),
    "Brown": (101, 67, 33),

    "Red": (220, 20, 60),
    "Burgundy": (128, 0, 32),
    "Pink": (255, 105, 180),
    "Light Pink": (255, 182, 193),
    "Coral": (255, 127, 80),

    "Orange": (255, 165, 0),
    "Yellow": (255, 215, 0),
    "Mustard": (204, 153, 0),

    "Green": (34, 139, 34),
    "Dark Green": (0, 100, 0),
    "Olive": (128, 128, 0),

    "Light Blue": (173, 216, 230),
    "Blue": (70, 130, 180),
    "Navy": (25, 25, 112),
    "Turquoise": (64, 224, 208),

    "Purple": (128, 0, 128),
    "Lavender": (216, 191, 216)
}

In [ ]:
# Function: Convert an RGB value to the nearest fashion color name
def get_fashion_color_name(rgb):

    # Extract the R, G, and B values
    r, g, b = map(int, rgb)

    closest_name = None
    min_distance = float("inf")

    # Compare the input color with each color in the fashion palette
    for color_name, color_rgb in fashion_colors.items():

        # Get the RGB values of the current palette color
        cr, cg, cb = color_rgb

        # Calculate the Euclidean distance between the two colors
        distance = np.sqrt(
            (r - cr) ** 2 +
            (g - cg) ** 2 +
            (b - cb) ** 2
        )

        # Keep the color with the smallest distance
        if distance < min_distance:
            min_distance = distance
            closest_name = color_name

    # Return the closest fashion color name
    return closest_name

In [ ]:
def rgb_to_hsv(rgb):

    # Convert the RGB color into a NumPy array with uint8 data type
    rgb_array = np.uint8([[rgb]])

    # Convert the color from RGB to HSV format
    hsv = cv2.cvtColor(
        rgb_array,
        cv2.COLOR_RGB2HSV
    )[0][0]

    # OpenCV represents Hue in the range 0-179
    # Convert it to 0-360 for easier use with a color wheel
    h = int(hsv[0]) * 2

    # Convert Saturation and Value from 0-255 to percentages
    s = int(hsv[1]) / 255 * 100
    v = int(hsv[2]) / 255 * 100

    return h, s, v

In [ ]:
for item in outfit_primary_colors:

    # Get the nearest fashion color name for the RGB value
    color_name = get_fashion_color_name(
        item["rgb"]
    )

    # Add the color name to the item information
    item["color_name"] = color_name

    # Display the item name, RGB value, and detected color name
    print(
        f'{item["item"]}: '
        f'RGB {item["rgb"]} -> '
        f'{color_name}'
    )

In [ ]:
def rgb_to_hsv(rgb):

    # Convert the RGB color into a NumPy array
    rgb_array = np.uint8([[rgb]])

    # Convert the color from RGB to HSV
    hsv = cv2.cvtColor(
        rgb_array,
        cv2.COLOR_RGB2HSV
    )[0][0]

    # Convert Hue from OpenCV's 0-179 range to 0-360 degrees
    h = int(hsv[0]) * 2

    # Convert Saturation and Value from 0-255 to percentages
    s = int(hsv[1]) / 255 * 100
    v = int(hsv[2]) / 255 * 100

    return h, s, v

In [ ]:
for item in outfit_primary_colors:

    # Convert the RGB color to HSV format
    hsv = rgb_to_hsv(item["rgb"])

    # Add the HSV values to the item information
    item["hsv"] = hsv

    # Display the item name, color name, RGB, and HSV values
    print(
        f'{item["item"]}: '
        f'{item["color_name"]} | '
        f'RGB {item["rgb"]} | '
        f'HSV {hsv}'
    )

In [ ]:
# Define a set of neutral fashion colors
neutral_colors = {
    "Black",
    "White",
    "Gray",
    "Dark Gray",
    "Light Gray",
    "Beige",
    "Cream"
}

In [ ]:
# Create separate lists for chromatic and neutral items
chromatic_items = []
neutral_items = []

for item in outfit_primary_colors:

    # Check whether the item's color is neutral
    if item["color_name"] in neutral_colors:
        neutral_items.append(item)

    else:
        # Add non-neutral colors to the chromatic items list
        chromatic_items.append(item)

In [ ]:
# Display the colors used for color harmony
print("Colors used for harmony:")

for item in chromatic_items:

    # Print the item name, color name, and hue value
    print(
        item["item"],
        "->",
        item["color_name"],
        "| Hue:",
        item["hsv"][0]
    )

# Display the neutral colors
print("\nNeutral colors:")

for item in neutral_items:

    # Print the item name and color name
    print(
        item["item"],
        "->",
        item["color_name"]
    )

Colors used for harmony:

Neutral colors:


In [ ]:
def analyze_color_harmony(chromatic_items, neutral_items):

    n = len(chromatic_items)

    # 1) All colors are neutral
    if n == 0:
        return "Neutral Harmony"

    # 2) One color + neutral colors
    if n == 1:
        if len(neutral_items) > 0:
            return "Neutral-Based Harmony"
        else:
            return "Single Color"

    # Extract the Hue values
    hues = [item["hsv"][0] for item in chromatic_items]

    # Calculate the shortest difference between two hues on the color wheel
    def hue_difference(h1, h2):
        diff = abs(h1 - h2)
        return min(diff, 360 - diff)

    # 3) If there are two colors
    if n == 2:

        diff = hue_difference(
            hues[0],
            hues[1]
        )

        if diff <= 15:
            return "Monochromatic"

        elif diff <= 45:
            return "Analogous"

        elif 150 <= diff <= 210:
            return "Complementary"

        else:
            return "No Standard Harmony"

    # 4) If there are three or more colors
    if n >= 3:

        # Use the first three colors for the current analysis
        h1, h2, h3 = hues[:3]

        d12 = hue_difference(h1, h2)
        d13 = hue_difference(h1, h3)
        d23 = hue_difference(h2, h3)

        differences = [d12, d13, d23]

        # Check for monochromatic harmony
        if max(differences) <= 15:
            return "Monochromatic"

        # Check for analogous harmony
        # The colors are relatively close to each other on the color wheel
        if max(differences) <= 60:
            return "Analogous"

        # Check for triadic harmony
        # Each pair of colors is approximately 120 degrees apart
        if all(90 <= d <= 150 for d in differences):
            return "Triadic"

        # Check if at least one pair of colors is complementary
        if any(150 <= d <= 210 for d in differences):
            return "Complementary-Based Harmony"

        return "No Standard Harmony"

In [ ]:
# Analyze the overall color harmony of the outfit
harmony = analyze_color_harmony(
    chromatic_items,
    neutral_items
)

# Display the detected outfit harmony
print("Outfit Harmony:", harmony)

Outfit Harmony: Neutral Harmony


In [ ]:
def get_color_score(harmony):

    # Assign a score to each type of color harmony
    scores = {
        "Neutral Harmony": 9.0,
        "Neutral-Based Harmony": 9.0,
        "Monochromatic": 9.0,
        "Analogous": 8.5,
        "Complementary": 9.0,
        "Triadic": 8.5,
        "Complementary-Based Harmony": 8.0,
        "Single Color": 7.5,
        "No Standard Harmony": 4.5
    }

    # Return the score for the detected harmony
    # Use 5.0 as the default score if the harmony type is not found
    return scores.get(harmony, 5.0)

In [ ]:
# Analyze the overall color harmony of the outfit
harmony = analyze_color_harmony(
    chromatic_items,
    neutral_items
)

# Calculate the color harmony score
color_score = get_color_score(harmony)

# Display the detected harmony and color score
print("Outfit Harmony:", harmony)
print("Color Score:", color_score, "/10")

Outfit Harmony: Neutral Harmony
Color Score: 9.0 /10
